# Feature construction and feature splitting
Rebuilt for pandas 3 and scikit-learn 1.9. Data: `data/titanic.csv`, the 891-passenger Titanic training file (full data: the Kaggle competition *Titanic: Machine Learning from Disaster*).

In [1]:
# libraries for tables, models and charts
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

## 1. Feature construction: family type

In [2]:
# keep four inputs and the target
df = pd.read_csv('data/titanic.csv')[['Age', 'Pclass', 'SibSp', 'Parch', 'Survived']]
# drop the 177 passengers with no age
df = df.dropna()
df.shape

(714, 5)

In [3]:
# inputs X and target y
X = df.drop(columns='Survived')
y = df['Survived']
X.head()

,Age,Pclass,SibSp,Parch
0,22.0,3,1,0
1,38.0,1,1,0
2,26.0,3,0,0
3,35.0,1,1,0
4,35.0,3,0,0


In [4]:
# baseline: mean accuracy over 20 folds (the same setup as the original code)
lr = LogisticRegression(max_iter=1000)
cross_val_score(lr, X, y, cv=20).mean()

np.float64(0.6933333333333332)

In [5]:
# family size: siblings/spouses + parents/children + the passenger
X_new = X.copy()
X_new['Family_size'] = X_new['SibSp'] + X_new['Parch'] + 1

# group the size: 0 = alone, 1 = small (2 to 4), 2 = large (5 or more)
size = X_new['Family_size']
X_new['Family_type'] = np.select([size == 1, size <= 4], [0, 1], 2)
X_new.head()

,Age,Pclass,SibSp,Parch,Family_size,Family_type
0,22.0,3,1,0,2,1
1,38.0,1,1,0,2,1
2,26.0,3,0,0,1,0
3,35.0,1,1,0,2,1
4,35.0,3,0,0,1,0


In [6]:
# survival rate for each family type: not a straight line
df.groupby(X_new['Family_type'])['Survived'].agg(['mean', 'size'])

,mean,size
Family_type,,
0,0.321782,404
1,0.579151,259
2,0.196078,51


In [7]:
# keep Family_type, drop the columns it was built from
X_new = X_new.drop(columns=['SibSp', 'Parch', 'Family_size'])
cross_val_score(lr, X_new, y, cv=20).mean()

np.float64(0.7003174603174602)

20 folds give a noisy number. Below we repeat 10-fold cross-validation 10 times (100 scores) and build every new column **inside a pipeline**, so each fold builds it from its own training rows.

In [8]:
# one function builds every new column from the raw columns
def add_columns(X):
    X = X.copy()
    X['Family_size'] = X['SibSp'] + X['Parch'] + 1
    size = X['Family_size']
    X['Family_type'] = np.select([size == 1, size <= 4], [0, 1], 2)
    # title: the text between ', ' and the first '.'
    X['Title'] = X['Name'].str.split(', ').str[1].str.split('.').str[0]
    X['Is_Married'] = (X['Title'] == 'Mrs').astype(int)
    return X

In [9]:
# a model = build columns -> keep (and one-hot encode) some -> logistic regression
def model(cols, onehot=()):
    pick = make_column_transformer(
        # titles seen fewer than 10 times are grouped into one 'infrequent' column
        (OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10), list(onehot)),
        ('passthrough', [c for c in cols if c not in onehot]))
    return make_pipeline(FunctionTransformer(add_columns), pick, LogisticRegression(max_iter=1000))

In [10]:
# the same 714 passengers for every comparison
raw = pd.read_csv('data/titanic.csv').dropna(subset=['Age'])
Xr = raw[['Age', 'Pclass', 'SibSp', 'Parch', 'Name']]
yr = raw['Survived']
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=0)

BASE = ['Age', 'Pclass', 'SibSp', 'Parch']
sets = {
    'baseline': model(BASE),
    '+ Family_size': model(BASE + ['Family_size']),
    'Family_type 0/1/2': model(['Age', 'Pclass', 'Family_type']),
    'Family_type one-hot': model(['Age', 'Pclass', 'Family_type'], ['Family_type']),
    '+ Is_Married': model(BASE + ['Is_Married']),
    '+ Title one-hot': model(BASE + ['Title'], ['Title']),
}
scores = {name: cross_val_score(m, Xr, yr, cv=cv) for name, m in sets.items()}
pd.DataFrame({'mean': [s.mean() for s in scores.values()],
              'std': [s.std() for s in scores.values()]}, index=scores.keys()).round(4)

,mean,std
baseline,0.6940,0.0478
+ Family_size,0.6940,0.0478
Family_type 0/1/2,0.7021,0.0476
Family_type one-hot,0.7274,0.0446
+ Is_Married,0.7594,0.0456
+ Title one-hot,0.8161,0.0404


In [11]:
# paired check: in how many of the 100 folds does a column beat the baseline?
for name in ['Family_type 0/1/2', 'Family_type one-hot', '+ Title one-hot']:
    d = scores[name] - scores['baseline']
    print(name, 'better:', (d > 0).sum(), 'worse:', (d < 0).sum(), 'tied:', (d == 0).sum())

Family_type 0/1/2 better: 51 worse: 29 tied: 20
Family_type one-hot better: 78 worse: 12 tied: 10
+ Title one-hot better: 99 worse: 0 tied: 1


In [12]:
# checks for Section 8: does Sex explain the title's gain, and does "Master" explain the rest?
# also the spread of the 20 single-fold scores of Section 3
Xs = raw[['Age', 'Pclass', 'SibSp', 'Parch']].copy()
s20 = cross_val_score(LogisticRegression(max_iter=1000), Xs, yr, cv=20)
print('20-fold: mean', round(s20.mean(), 4), ' std', round(s20.std(), 4))
Xs['Sex'] = (raw['Sex'] == 'male').astype(int)
print('baseline + Sex', cross_val_score(LogisticRegression(max_iter=1000), Xs, yr, cv=cv).mean().round(4))
title = raw['Name'].str.split(', ').str[1].str.split('.').str[0]
Xs['Master'] = (title == 'Master').astype(int)
print('baseline + Sex + Master', cross_val_score(LogisticRegression(max_iter=1000), Xs, yr, cv=cv).mean().round(4))

20-fold: mean 0.6933  std 0.0913


baseline + Sex 0.8049


baseline + Sex + Master 0.829


## 2. Feature splitting: the title inside the name

In [13]:
df = pd.read_csv('data/titanic.csv')
df['Name'].head()

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
Name: Name, dtype: str

In [14]:
# step 1: split at ', ' and keep piece 1; step 2: split that at '.' and keep piece 0
df['Title'] = df['Name'].str.split(', ').str[1].str.split('.').str[0]
df[['Title', 'Name']].head()

,Title,Name
0,Mr,"Braund, Mr. Owen Harris"
1,Mrs,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,Miss,"Heikkinen, Miss. Laina"
3,Mrs,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,Mr,"Allen, Mr. William Henry"


In [15]:
# survival rate and passenger count for each title
# (select the column first: pandas 3 cannot average text columns)
rate = df.groupby('Title')['Survived'].agg(['mean', 'size'])
rate.sort_values('size', ascending=False)

,mean,size
Title,,
Mr,0.156673,517
Miss,0.697802,182
Mrs,0.792000,125
Master,0.575000,40
Dr,0.428571,7
Rev,0.000000,6
Mlle,1.000000,2
Major,0.500000,2
Col,0.500000,2


In [16]:
fig = px.bar(rate.sort_values('size').reset_index(), x='mean', y='Title', orientation='h',
             hover_data=['size'], labels={'mean': 'survival rate'})
fig.show()

In [17]:
# a new 0/1 column from the title, in one step
# (the old two-step df['Is_Married'].loc[...] = 1 changes nothing in pandas 3)
df['Is_Married'] = (df['Title'] == 'Mrs').astype(int)
df['Is_Married'].sum()

np.int64(125)